# Лабораторная 2 Часть 2 Анализ интернет магазина

**Иноземцев Сергей Викторович · ЕТ-216**

Источник задания: «лб2 (ч2).pdf». Все вычисления воспроизводимы при последовательном запуске ячеек. Фиксированные random seed заданы в коде.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
DATA = Path('.')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight')
    plt.show()
    plt.close('all')


## 0 Генерация 3000 заказов
Учебная симуляция, а не реальные продажи. cost — полная себестоимость заказа, чтобы profit=revenue−cost имел согласованные единицы.

In [ ]:
np.random.seed(2024);n=3000
cities=['Москва','Санкт-Петербург','Казань','Екатеринбург','Челябинск'];categories=['Электроника','Одежда','Дом','Красота','Спорт']
cat=np.random.choice(categories,n);price=np.round(np.random.lognormal(7.5,.9,n),2);quantity=np.random.randint(1,7,n)
df=pd.DataFrame({'order_id':np.arange(1,n+1),'order_date':pd.Timestamp('2023-01-01')+pd.to_timedelta(np.random.randint(0,731,n),unit='D'),
'customer_id':np.random.randint(1,1201,n),'city':np.random.choice(cities,n),'age':np.random.randint(16,76,n).astype(float),
'is_vip':np.random.choice([True,False],n,p=[.2,.8]),'product_name':np.char.add(cat.astype(str),np.random.choice([' A',' B',' C',' D'],n)),
'category':cat,'quantity':quantity,'price':price,'discount':np.random.choice([0,.05,.1,.2,.25,.3],n),
'cost':np.round(price*quantity*np.random.uniform(.55,.95,n),2),'status':np.random.choice(['Доставлен','Отменён','Возврат'],n,p=[.8,.12,.08]),
'payment':np.random.choice(['Карта','Онлайн','Наличные'],n)})
df.loc[np.random.choice(n,30,replace=False),'age']=np.nan
df.to_csv('orders_raw.csv',index=False)
display(df.head());display(df.tail());print('Размер:',df.shape);df.info();display(df.describe());display(df.isna().sum())
print('Уникальных клиентов/товаров:',df.customer_id.nunique(),df.product_name.nunique())
for c in ['city','category','payment']:print(c,df[c].unique())
assert df.age.isna().sum()==30 and len(df)==3000

## 2 Подготовка

In [ ]:
df['order_date']=pd.to_datetime(df.order_date);df['year']=df.order_date.dt.year;df['month']=df.order_date.dt.month
df['revenue']=df.price*df.quantity*(1-df.discount);df['profit']=df.revenue-df.cost
df['is_real_sale']=df.status.eq('Доставлен')
median_age=df.age.median();df['age']=df.age.fillna(median_age)
df['age_group']=pd.cut(df.age,bins=[0,18,30,45,60,100],labels=['до 18','18–30','31–45','46–60','60+'])
display(df.age_group.value_counts(sort=False));print('Пропусков после:',df.age.isna().sum(),'Медиана:',median_age)
print('Убыточных заказов:',(df.profit<0).sum());display(df.head())
df.to_csv('orders_prepared.csv',index=False)

Прибыль может быть отрицательна, если скидка уменьшила выручку ниже полной себестоимости. Для фактической выручки далее используются только доставленные заказы; отменённые и возвращённые не считаются реализованными продажами. Группы возраста воспроизводят границы из условия: интервал с подписью «до 18» включает 18, следующий — (18,30]. Возраст задан на уровне заказа; при реальной работе его следовало бы согласовать по клиенту.

## 3 Все фильтры и экстремальные значения

In [ ]:
filters={'quantity>3':df.quantity>3,'Москва':df.city.eq('Москва'),'VIP':df.is_vip,'price>5000':df.price>5000,
'discount>20%':df.discount>.2,'Москва и revenue>5000':df.city.eq('Москва')&(df.revenue>5000),
'VIP Электроника':df.is_vip&df.category.eq('Электроника'),'Доставлен Онлайн':df.is_real_sale&df.payment.eq('Онлайн'),
'2024':df.year.eq(2024),'Январь 2024':df.year.eq(2024)&df.month.eq(1)}
for name,mask in filters.items():
    print(name,'—',mask.sum());display(df.loc[mask].head())
display(df.status.value_counts())
for col,kind in [('revenue','max'),('discount','max'),('profit','max'),('age','min'),('age','max')]:
    idx=getattr(df[col],'idx'+kind)();print(col,kind);display(df.loc[[idx]])

## 4 Группировки
Сначала показатели всех заказов по условию, затем отдельно показатели фактических продаж.

In [ ]:
display(df.groupby('category').revenue.sum());display(df.groupby('category').revenue.mean());display(df.groupby('category').size());display(df.groupby('category').discount.mean())
display(df.groupby(['category','city']).revenue.sum())
display(df.loc[df.is_vip].groupby('city').revenue.mean())
display(df.groupby(['city','payment']).size().unstack(fill_value=0))
summary=df.groupby('category').agg(orders=('order_id','count'),revenue=('revenue','sum'),avg_revenue=('revenue','mean'),avg_discount=('discount','mean'),profit=('profit','sum'));display(summary)
real=df.loc[df.is_real_sale].copy();actual=real.groupby('category').revenue.sum().sort_values(ascending=False);display(actual)

## 5 Девять визуализаций

In [ ]:
fig,ax=plt.subplots(3,3,figsize=(17,13))
for a,col,title in zip(ax.flat[:3],['age','revenue','discount'],['Возраст','Расчётная выручка заказа','Скидка']):
    a.hist(df[col],bins=20);a.set(title=title,xlabel=col,ylabel='Число заказов')
ax[1,0].hist([df.loc[df.is_vip,'age'],df.loc[~df.is_vip,'age']],label=['VIP','Обычные'],bins=10);ax[1,0].legend();ax[1,0].set(title='Возраст VIP и обычных',xlabel='Возраст',ylabel='Заказы')
actual.plot.bar(ax=ax[1,1],rot=40);ax[1,1].set(title='Фактическая выручка категорий',xlabel='Категория',ylabel='Рубли')
df.city.value_counts().plot.bar(ax=ax[1,2],rot=40);ax[1,2].set(title='Заказы по городам',xlabel='Город',ylabel='Заказы')
df.age_group.value_counts(sort=False).plot.bar(ax=ax[2,0],rot=30);ax[2,0].set(title='Возрастные группы',xlabel='Группа',ylabel='Заказы')
df.status.value_counts().plot.bar(ax=ax[2,1],rot=30);ax[2,1].set(title='Статусы',xlabel='Статус',ylabel='Заказы')
monthly=real.groupby(real.order_date.dt.to_period('M')).revenue.sum();monthly.index=monthly.index.to_timestamp();monthly.plot(ax=ax[2,2]);ax[2,2].set(title='Фактическая выручка по месяцам',xlabel='Месяц',ylabel='Рубли')
show('store_dashboard')

## 6 Мини анализ

In [ ]:
vip_share=real.loc[real.is_vip,'revenue'].sum()/real.revenue.sum()
print('Всего заказов:',len(df),'Доставлено:',len(real),'Расчётная сумма всех:',df.revenue.sum(),'Фактическая:',real.revenue.sum(),'Средняя доставленного:',real.revenue.mean())
print('Максимум/минимум категорий:',actual.idxmax(),actual.idxmin(),'Доля VIP:',vip_share)
df['discount_group']=pd.cut(df.discount,[-.001,.1,.2,.31],labels=['0–10%','>10–20%','>20–30%'])
discount_stats=df.groupby('discount_group',observed=True).revenue.agg(['count','mean']);display(discount_stats)
losses=pd.crosstab(df.category,df.status).reindex(columns=['Отменён','Возврат'],fill_value=0);losses['Всего']=losses.sum(axis=1);display(losses)

Всего 3000 заказов, из них 2403 доставлены; отмены и возвраты исключены из фактической выручки 19,170,545.94 руб. Средняя выручка доставленного заказа 7977.76 руб. Наибольшая фактическая выручка у категории «Одежда», наименьшая — «Дом». Самое большое число заказов у города «Казань». VIP дают 19.0% фактической выручки. Сравнение скидочных групп наблюдательное: различия среднего чека зависят также от цены и количества, причинный эффект скидки не установлен. Больше всего отмен и возвратов в абсолютных числах у «Красота»; дополнительно нужны доли от заказов категории, анализ маржи и проверка результатов на реальных данных.

## Контрольные вопросы
1. DataFrame — двумерная таблица с именованными столбцами и индексом.
2. loc выбирает по меткам (правая метка среза включается), iloc — по позициям (правая граница исключается).
3. groupby разделяет наблюдения на группы; agg задаёт одно или несколько агрегирований.
4. mean чувствителен к экстремумам, median — центральная порядковая величина.
5. Пропуски находятся isna; методы: удаление, константа, среднее/медиана/мода, прогнозное заполнение.
6. pd.cut разбивает числовой признак на интервалы; value_counts считает частоты; idxmax возвращает индекс первого максимума.
7. Несколько фильтров объединяются &, | и скобками; качество данных проверяется до анализа, чтобы избежать искажённых выводов.
8. revenue — выручка, profit — выручка минус затраты. Отмены и возвраты не являются реализованной выручкой в этой модели.
9. Визуализация обнаруживает распределения и зависимости; Git хранит версии, позволяет видеть изменения и воспроизводить этапы работы.